# Autoencoder OOD Detection (8 Points)

Train a **single convolutional autoencoder** on UC Merced.
Use **reconstruction MSE** as the OOD score:

- UC Merced test → AE reconstructs well → low MSE
- FIDS30 → AE has never seen fruits → high MSE

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

IMG_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder("PrepData/UC_Merced/Training", transform=transform)
val_ds = datasets.ImageFolder("PrepData/UC_Merced/Validation", transform=transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

## 1. Autoencoder Architecture (provided)

- Encoder: three stride-2 conv layers, channels `3 → 32 → 64 → 128`, spatial 256 → 32
- Decoder: three stride-2 transposed convs, channels `128 → 64 → 32 → 3`, spatial 32 → 256
- Final `Sigmoid` so outputs lie in [0, 1]

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = Autoencoder().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Train on UC Merced

Standard reconstruction training: input = target.

In [ ]:
# TODO: Choose the loss (MSELoss) and optimizer (Adam with lr=LR).
criterion = ___
optimizer = ___

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        images = images.to(device)
        # TODO: forward pass and compute loss (autoencoder: target = input)
        loss = criterion(model(images), ___)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, _ in val_loader:
            images = images.to(device)
            val_loss += criterion(model(images), images).item()
    print(f"Epoch {epoch+1}: Train Loss={train_loss/len(train_loader):.5f}, Val Loss={val_loss/len(val_loader):.5f}")

torch.save(model.state_dict(), "autoencoder.pth")
print("Saved to autoencoder.pth")

## 3. Reconstruction MSE: UC Merced (test) vs FIDS30

In [ ]:
uc_test_ds = datasets.ImageFolder("PrepData/UC_Merced/Test", transform=transform)
fids_ds = datasets.ImageFolder("PrepData/FIDS30", transform=transform)
uc_loader = DataLoader(uc_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
fids_loader = DataLoader(fids_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

def compute_errors(model, loader):
    """Per-sample MSE + original + reconstructed images (provided)."""
    errors, all_images, all_recons = [], [], []
    model.eval()
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            recon = model(images)
            mse = (images - recon).pow(2).mean(dim=[1, 2, 3])
            errors.append(mse.cpu())
            all_images.append(images.cpu())
            all_recons.append(recon.cpu())
    return torch.cat(errors).numpy(), torch.cat(all_images), torch.cat(all_recons)

# TODO: Call compute_errors on both loaders.
uc_errors, uc_images, uc_recons = ___
fids_errors, fids_images, fids_recons = ___

print(f"UC Merced: mean MSE = {uc_errors.mean():.5f}")
print(f"FIDS30:    mean MSE = {fids_errors.mean():.5f}")
print(f"Ratio:     {fids_errors.mean() / uc_errors.mean():.2f}x")

## 4. Histogram + Threshold Analysis

In [ ]:
# TODO: Same threshold analysis as in notebook 01, but using MSE instead of variance.
threshold = np.percentile(uc_errors, ___)
false_alarm = (uc_errors > threshold).mean()
detection = (___ > threshold).mean()

print(f"Threshold (95th pct of UC Merced MSE): {threshold:.5f}")
print(f"  False alarm rate: {false_alarm:.1%}")
print(f"  Detection rate:   {detection:.1%}")

# Histogram (provided)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(uc_errors, bins=40, alpha=0.7, label=f"UC Merced (n={len(uc_errors)})", color='steelblue')
ax.hist(fids_errors, bins=40, alpha=0.7, label=f"FIDS30 (n={len(fids_errors)})", color='tomato')
ax.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f"Threshold = {threshold:.4f}")
ax.set_xlabel("Reconstruction MSE")
ax.set_ylabel("Count")
ax.set_title(f"Autoencoder OOD: {detection:.0%} detection, {false_alarm:.0%} false alarm")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Visual Comparison: Reconstructions (Run and get result for report)

In [ ]:
def show_recons(images, recons, errors, title, n=6):
    sorted_idx = np.argsort(errors)
    pick = sorted_idx[np.linspace(0, len(sorted_idx)-1, n, dtype=int)]
    fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 5))
    for j, idx in enumerate(pick):
        axes[0, j].imshow(images[idx].permute(1, 2, 0).clamp(0, 1)); axes[0, j].axis('off')
        axes[1, j].imshow(recons[idx].permute(1, 2, 0).clamp(0, 1)); axes[1, j].axis('off')
        axes[1, j].set_xlabel(f"MSE={errors[idx]:.4f}", fontsize=8)
    axes[0, 0].set_ylabel("Input", fontsize=11)
    axes[1, 0].set_ylabel("Reconstruction", fontsize=11)
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

show_recons(uc_images, uc_recons, uc_errors, "UC Merced (in-distribution)")
show_recons(fids_images, fids_recons, fids_errors, "FIDS30 (out-of-distribution)")